# Classical Baseline Models

## 1. Classification Task


This notebook compares Logistic Regression, SVM, and Random Forest for binary chest-X-ray classification: **Pneumonia** (`Lung Opacity`, label 1) versus **Healthy** (`Normal`, label 0). The local `No Lung Opacity / Not Normal` class was excluded during Stage 1.

The available labeled cohort contains 14,863 images rather than the paper's reported 17,489-image cohort. The results are measured on our fixed split and are not an exact cohort-matched reproduction.

## 2. Dataset and fixed split

All three experiments use the same permanent, seed-42 stratified assignments in `metadata/split_metadata.csv`. The RSNA `image_id` is also the patient ID here; each appears in only one split.

In [ ]:
import csv
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "metadata").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "metadata").is_dir():
    raise FileNotFoundError("Could not locate project metadata from notebook working directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with (ROOT / "metadata" / "split_metadata.csv").open(newline="", encoding="utf-8-sig") as f:
    split_rows = list(csv.DictReader(f))
split_counts = {}
for split in ("train", "validation", "test"):
    rows = [row for row in split_rows if row["split"] == split]
    split_counts[split] = {
        "total": len(rows),
        "pneumonia": sum(row["label"] == "1" for row in rows),
        "healthy": sum(row["label"] == "0" for row in rows),
    }
assert len({row["image_id"] for row in split_rows}) == len(split_rows)
display(Markdown("| Split | Images | Pneumonia | Healthy | Share |\n|---|---:|---:|---:|---:|\n" + "\n".join(
    f"| {name} | {counts['total']:,} | {counts['pneumonia']:,} | {counts['healthy']:,} | {100 * counts['total'] / len(split_rows):.2f}% |"
    for name, counts in split_counts.items()
)))

### Preprocessing and representation

Original DICOM images are decoded on demand, resized to **128 ? 128** grayscale pixels, scaled to [0, 1], and normalized using train-only global mean and standard deviation. Each classical model receives the same **16,384 flattened normalized pixels**, through the existing shared dataset and preprocessing code.

In [ ]:
preprocessing = json.loads((ROOT / "metadata" / "preprocessing_stats.json").read_text(encoding="utf-8"))
print(f"Image dimensions: {preprocessing['image_size'][0]} ? {preprocessing['image_size'][1]}")
print(f"Train-only pixel mean: {preprocessing['mean']:.8f}")
print(f"Train-only pixel standard deviation: {preprocessing['std']:.8f}")
print(f"Statistics fitted using {preprocessing['fit_image_count']:,} training images")
print(f"Feature count per image: {preprocessing['image_size'][0] * preprocessing['image_size'][1]:,}")

## 3. Logistic Regression

Stage 5's actual result is loaded from disk; this notebook does **not** retrain Logistic Regression. Reference results supplied for the paper: train 75.86%, test 73.02%.

In [ ]:
logistic_result = json.loads((ROOT / "results" / "logistic_regression.json").read_text(encoding="utf-8"))
logistic_metrics = logistic_result["metrics"]
logistic_reference = logistic_result["reference_result_percent"]
print(f"Reference ? train: {logistic_reference['train']:.2f}%, test: {logistic_reference['test']:.2f}%")
print(f"Measured ? train: {logistic_metrics['train_accuracy_percent']:.2f}%, validation: {logistic_metrics['validation_accuracy_percent']:.2f}%, test: {logistic_metrics['test_accuracy_percent']:.2f}%")

## 4. SVM

The reference summary does not give the kernel or hyperparameters. The recorded SVM uses a linear kernel and C = 1.0. Its saved metrics are loaded below; SVM is not retrained.

In [ ]:
svm_result = json.loads((ROOT / "results" / "svm.json").read_text(encoding="utf-8"))
svm_metrics = svm_result["metrics"]
svm_reference = svm_result["reference_result"]
print(f"Configuration: {svm_result['configuration']['kernel']} kernel, C={svm_result['configuration']['c']}")
print(f"Reference ? train: {svm_reference['train_accuracy_percent']:.2f}%, test: {svm_reference['test_accuracy_percent']:.2f}%")
print(f"Measured ? train: {svm_metrics['train_accuracy_percent']:.2f}%, validation: {svm_metrics['validation_accuracy_percent']:.2f}%, test: {svm_metrics['test_accuracy_percent']:.2f}%")

## 5. Random Forest

The supplied paper summary reports results but no Random Forest hyperparameters. Our deterministic, bounded-depth configuration uses 100 trees, Gini impurity, `max_features="sqrt"`, maximum depth 20, and minimum leaf size 2. Training fits only the train split; validation is reported as evaluation and the test split is evaluated after fitting.

The next cell loads the completed run by default. Set `RUN_RANDOM_FOREST = True` to execute a fresh fit using the reusable training pipeline.

In [ ]:
RUN_RANDOM_FOREST = False
if RUN_RANDOM_FOREST:
    from scripts.train_random_forest import run as run_random_forest
    from src.models.random_forest import RandomForestConfig
    random_forest_result = run_random_forest(RandomForestConfig())
else:
    random_forest_result = json.loads((ROOT / "results" / "random_forest.json").read_text(encoding="utf-8"))

rf_metrics = random_forest_result["metrics"]
print("Configuration:", random_forest_result["configuration"])
print(f"Reference ? train: {random_forest_result['reference_result']['train_accuracy_percent']:.2f}%, test: {random_forest_result['reference_result']['test_accuracy_percent']:.2f}%")
print(f"Measured ? train: {rf_metrics['train_accuracy_percent']:.2f}%, validation: {rf_metrics['validation_accuracy_percent']:.2f}%, test: {rf_metrics['test_accuracy_percent']:.2f}%")

## 6. Final comparison

Differences are our measured test accuracy minus the paper's reference test accuracy, reported in percentage points.

In [ ]:
with (ROOT / "results" / "classification_results.csv").open(newline="", encoding="utf-8-sig") as f:
    comparison_rows = list(csv.DictReader(f))
expected_models = {"Logistic Regression", "SVM", "Random Forest"}
assert {row["model"] for row in comparison_rows} == expected_models

table = "| Model | Reference Test Accuracy | Our Test Accuracy | Difference |\n|---|---:|---:|---:|\n"
for row in comparison_rows:
    reference = float(row["reference_test_accuracy"])
    actual = float(row["test_accuracy"])
    difference = float(row["difference_from_reference"])
    table += f"| {row['model']} | {reference:.2%} | {actual:.2%} | {difference * 100:+.2f} pp |\n"
display(Markdown(table))

In [ ]:
from PIL import Image as PILImage, ImageDraw

chart = PILImage.new("RGB", (780, 330), "white")
draw = ImageDraw.Draw(chart)
left, right, top = 200, 735, 55
draw.text((left, 18), "Reference accuracy vs measured accuracy (%)", fill="#222222")
for tick in range(0, 101, 20):
    x = left + (right - left) * tick / 100
    draw.line((x, top - 5, x, 300), fill="#e0e0e0", width=1)
    draw.text((x - 8, 305), str(tick), fill="#333333")
for index, row in enumerate(comparison_rows):
    y = top + index * 74
    draw.text((12, y + 15), row["model"], fill="#222222")
    for offset, key, color in ((0, "reference_test_accuracy", "#9aa5b1"), (25, "test_accuracy", "#2b6cb0")):
        value = float(row[key]) * 100
        x_end = left + (right - left) * value / 100
        draw.rectangle((left, y + offset, x_end, y + offset + 17), fill=color)
        draw.text((min(x_end + 5, right - 32), y + offset), f"{value:.1f}%", fill="#222222")
draw.rectangle((left, 285, left + 14, 298), fill="#9aa5b1")
draw.text((left + 20, 284), "Reference", fill="#222222")
draw.rectangle((left + 130, 285, left + 144, 298), fill="#2b6cb0")
draw.text((left + 150, 284), "Our result", fill="#222222")
display(chart)

## 7. Discussion

Random Forest performed best on this fixed split (87.15% test accuracy), followed by SVM (83.58%) and Logistic Regression (83.24%). The result differs from the paper's 83.00% Random Forest reference; our usable cohort has 14,863 rather than 17,489 images, and the paper summary does not provide the Random Forest settings, so this is not a like-for-like reproduction. Flattened pixels discard an explicit spatial hierarchy and produce a high-dimensional feature vector; this is a limitation of the representation, but these results alone do not establish that it caused the performance differences.